# 01 — Volatility Dynamics baseline from first principles

## Independence contract

This notebook is intentionally independent of every other research branch.

It **does not import or consume**:

- HSMM outputs, code, state definitions, HAR models, or frozen specifications;
- Volatility Regression outputs, models, selected features, or confirmation logic;
- Volatility Risk Premium outputs or option-pricing conventions;
- Pairwise Feature Regression outputs, correlations, clock findings, or feature selections.

The only project input is the shared research database relation `time_series_feature_state`, and Module 01 reads only three base columns:

`state_time`, `underlying`, `return_5m`.

All volatility objects, thresholds, clock diagnostics, persistence statistics and episode definitions are constructed inside this branch from those columns.

## Research question

Before forecasting volatility or imposing latent states, what does the volatility process itself look like?

Module 01 establishes:

1. realized-variance distributions at several backward horizons;
2. whether a deterministic UTC clock pattern exists **in this branch's own data**;
3. persistence of volatility and association between current and subsequent realized variance;
4. duration and shape of high-volatility episodes;
5. buildup and decay around the onset of those episodes.

No MOM, MR, VS, Bridge, HMM, HSMM, or predictive-state label is used here.


In [ ]:
from pathlib import Path
import json, os, sys, warnings
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 260)
pd.set_option('display.max_rows', 300)

load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.modeling import volatility_dynamics as vd

START = os.getenv('VD_START', '').strip() or None
END = os.getenv('VD_END', '').strip() or None
ASSET_FILTER = [x.strip().upper() for x in os.getenv('VD_ASSETS', '').split(',') if x.strip()]
HIGH_VOL_Q = float(os.getenv('VD_HIGH_VOL_Q', '0.90'))
OUT = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics' / '01'
OUT.mkdir(parents=True, exist_ok=True)

print('volatility_dynamics', vd.VOLATILITY_DYNAMICS_VERSION)
print('selftest:', vd.selftest())
print('window:', START or 'earliest available', '->', END or 'latest available')
print('high-vol threshold quantile:', HIGH_VOL_Q)


## 1. Load only timestamp, asset and raw 5-minute index return

No research-branch feature is read. Optional start/end filters exist only for reproducibility; the default is the entire available database history.


In [ ]:
con = duckdb.connect(str(DB_PATH), read_only=True)
available = set(con.execute('DESCRIBE time_series_feature_state').df()['column_name'])
required = {'state_time','underlying','return_5m'}
missing = sorted(required - available)
if missing:
    raise RuntimeError(f'time_series_feature_state lacks required columns: {missing}')

where = []
if START:
    where.append(f"state_time >= TIMESTAMPTZ '{pd.Timestamp(START, tz='UTC').isoformat()}'")
if END:
    where.append(f"state_time <= TIMESTAMPTZ '{(pd.Timestamp(END, tz='UTC') + pd.Timedelta(days=1)).isoformat()}'")
if ASSET_FILTER:
    where.append('underlying IN (' + ','.join(repr(x) for x in ASSET_FILTER) + ')')
where_sql = '' if not where else ' WHERE ' + ' AND '.join(where)
df = con.execute(
    'SELECT state_time, underlying, return_5m FROM time_series_feature_state'
    + where_sql + ' ORDER BY underlying, state_time'
).df()
con.close()
df['state_time'] = pd.to_datetime(df['state_time'], utc=True)
if df.empty:
    raise RuntimeError('No rows found for requested data window.')
dupes = int(df.duplicated(['underlying','state_time']).sum())
if dupes:
    raise RuntimeError(f'Found {dupes} duplicate asset/timestamp rows; resolve data integrity before research.')
print(f'rows {len(df):,} | assets {sorted(df.underlying.unique())}')
print(df.state_time.min(), '->', df.state_time.max())


## 2. Construct volatility objects independently

For a backward window of (k) contiguous five-minute returns,

\[
RV^{back}_{t,k}=\sum_{j=0}^{k-1} r_{t-j}^2,
\qquad
Rate^{back}_{t,k}=RV^{back}_{t,k}/k.
\]

For descriptive current-versus-future association only,

\[
RV^{fwd}_{t,k}=\sum_{j=1}^{k} r_{t+j}^2.
\]

Every window requires a complete exact timestamp grid; windows crossing gaps are missing rather than rescaled.


In [ ]:
panels, metas, eps_tables = [], [], []
for asset, g in df.groupby('underlying', sort=True):
    p, meta, eps = vd.build_volatility_panel(g)
    panels.append(p)
    metas.append(meta.__dict__)
    eps_tables.append(eps)
    print(asset, '| bar minutes', meta.bar_minutes, '| rows', meta.n_rows, '| finite return share', round(meta.finite_return_share,4))
panel = pd.concat(panels, ignore_index=True)
panel_meta = pd.DataFrame(metas)
epsilons = pd.concat(eps_tables, ignore_index=True)
coverage = vd.data_coverage(panel)
dist = vd.rv_distribution(panel)
display(panel_meta)
display(coverage)


## 3. Distribution of realized volatility

This is the first empirical object in the branch. We inspect scale, skew and tail concentration before any state or forecasting assumption.


In [ ]:
display(dist.round(10))

for asset in sorted(panel.underlying.unique()):
    x = panel.loc[panel.underlying == asset, 'log_back_rate_30m'].dropna()
    if len(x):
        fig, ax = plt.subplots(figsize=(8,3))
        ax.hist(x, bins=60)
        ax.set_title(f'{asset}: log backward 30m variance rate')
        ax.set_xlabel('log variance per 5m bar')
        plt.show()


## 4. Does this branch independently find UTC clock structure?

We do not assume the answer. For each exact five-minute UTC slot, we compute the median log backward 30m variance rate across dates, then compare the cross-clock amplitude with the pooled interquartile range.


In [ ]:
clock_profile, clock_summary = vd.clock_profile(panel)
display(clock_summary.round(4))

for asset in sorted(panel.underlying.unique()):
    g = clock_profile[clock_profile.underlying == asset].sort_values('clock_slot')
    if len(g):
        fig, ax = plt.subplots(figsize=(10,3))
        ax.plot(g.clock_slot, g['median'])
        ax.set_title(f'{asset}: internally measured UTC clock profile')
        ax.set_xlabel('5-minute UTC clock slot (0..287)')
        ax.set_ylabel('median log backward 30m variance rate')
        plt.show()


## 5. Volatility persistence from raw returns

We report overlapping autocorrelations and current-backward-vs-future associations, plus non-overlapping 30m/60m origin subsets. These are descriptive dependence statistics, not a predictive model.


In [ ]:
persistence = vd.persistence_summary(panel)
display(persistence.pivot(index='underlying', columns='relationship', values='corr').round(3))


## 6. High-volatility episodes, defined without a forecast model

For each asset, an episode is a contiguous run where the backward 30m variance rate is at or above that asset's full-sample 90th percentile (configurable with `VD_HIGH_VOL_Q`).

This is a retrospective descriptive threshold chosen inside this branch. It is **not** a live state definition and it is not adjusted for clock in Module 01. The clock analysis above tells us whether a later module needs that adjustment.


In [ ]:
thresholds = vd.high_vol_thresholds(panel, q=HIGH_VOL_Q)
episodes = vd.extract_high_vol_episodes(panel, thresholds)
episode_summary = vd.episode_summary(episodes)
display(thresholds)
display(episode_summary.round(4))


## 7. Buildup and decay around episode onset

The anchor is the first bar crossing into the high-volatility region. Paths use volatility measures derived only from returns. No flow, IV, order-book, or cross-branch feature is examined yet.


In [ ]:
paths = vd.event_path_summary(panel, episodes, pre_min=120, post_min=180)
print('path summary rows:', len(paths))

for asset in sorted(panel.underlying.unique()):
    g = paths[(paths.underlying == asset) & (paths.variable == 'log_back_rate_30m')].sort_values('rel_min')
    if len(g):
        fig, ax = plt.subplots(figsize=(9,3))
        ax.plot(g.rel_min, g['median'], marker='o', markersize=2)
        ax.fill_between(g.rel_min, g.q25, g.q75, alpha=0.2)
        ax.axvline(0, linewidth=1)
        ax.set_title(f'{asset}: rolling 30m volatility around high-vol episode onset')
        ax.set_xlabel('minutes relative to onset')
        ax.set_ylabel('log backward 30m variance rate')
        plt.show()


## 8. Persist compact, branch-local outputs

No full state panel is committed. Everything below can be rebuilt from the base database and this standalone engine.


In [ ]:
panel_meta.to_csv(OUT / '01_panel_metadata.csv', index=False)
epsilons.to_csv(OUT / '01_log_epsilons.csv', index=False)
coverage.to_csv(OUT / '01_data_coverage.csv', index=False)
dist.to_csv(OUT / '01_rv_distribution.csv', index=False)
clock_profile.to_csv(OUT / '01_clock_profile.csv', index=False)
clock_summary.to_csv(OUT / '01_clock_effect_summary.csv', index=False)
persistence.to_csv(OUT / '01_persistence.csv', index=False)
thresholds.to_csv(OUT / '01_high_vol_thresholds.csv', index=False)
episodes.to_csv(OUT / '01_high_vol_episodes.csv', index=False)
episode_summary.to_csv(OUT / '01_high_vol_episode_summary.csv', index=False)
paths.to_csv(OUT / '01_high_vol_paths.csv', index=False)

meta = {
    'version': vd.VOLATILITY_DYNAMICS_VERSION,
    'independence_contract': 'No imports or outputs from any other research branch; only state_time, underlying, return_5m from time_series_feature_state.',
    'database_relation': 'time_series_feature_state',
    'database_columns_read': ['state_time','underlying','return_5m'],
    'database_read_only': True,
    'start': START or 'earliest_available',
    'end': END or 'latest_available',
    'assets': sorted(panel.underlying.unique().tolist()),
    'n_rows': int(len(panel)),
    'high_vol_quantile': HIGH_VOL_Q,
    'episode_definition': 'contiguous backward-30m variance-rate exceedance; retrospective descriptive only',
    'no_predictive_model': True,
    'no_external_research_assumptions': True,
}
(OUT / '01_run_metadata.json').write_text(json.dumps(meta, indent=2))
print('wrote independent Module 01 outputs to', OUT)


## Interpretation boundary

Module 01 can establish empirical facts about volatility distribution, deterministic clock structure, persistence, and high-volatility episode shape.

It cannot yet establish:

- that volatility is forecastable by a particular model;
- an expected-vs-surprise decomposition;
- a Volatility Shock latent state;
- Bridge/absorption behavior;
- MOM or MR behavior;
- HMM or HSMM duration structure.

Those questions must be built sequentially from results produced **inside this branch**.
